목적

L&E LLM Edit without Original가 왜 세희님이 진행한 preliminary experiment 결과와 다르게 (다만, 그 때는 SNLI 데이터를 고친 것이었음)
LLM Edit without Locate이나 L&E LLM Edit with Original 보다 contradiction rate 가 높다고 나온걸까?

분석 방법

에러 분석을 통해서, 어떤 사유로 더 빈번히 틀리는지 확인

그냥 눈으로 봐서는 더 많이 틀리는 양상 파악이 좀 힘들다.

In [1]:
from new_module.dev_utils.utils import *
import random
from pprint import pp

print=pp

In [2]:
# 원본 결과 및 원본의 NLI 결과 읽어오기

original = read_outputs('new_module/data/logical-consistency/anli-r2-test_prompt_4_below_consistent_threshold_3105.jsonl')
original_sat = read_metric_file('new_module/data/logical-consistency/anli-r2-test_prompt_4_below_consistent_threshold_3105.jsonl-results.txt.nli', 'nli')

In [40]:
# output 파일 읽어오기
output_nli = read_outputs('/data/hyeryung/mucoco/outputs/llmedit/results_2025/edited_nli/6_nli_edited_52941.jsonl_total_0')
output_fluency = read_outputs('/data/hyeryung/mucoco/outputs/llmedit/results_2025/edited_fluency/6_nli_edited_52941.jsonl_total_0')
output = read_outputs('/data/hyeryung/mucoco/outputs/llmedit/results_2025/edited/6_nli_edited_52941.jsonl_total_0')
located = read_outputs('/data/hyeryung/mucoco/outputs/llmedit/results_2025/located/6_nli_located_52941.jsonl_filtered_0')
located_both = read_outputs('/data/hyeryung/mucoco/outputs/llmedit/dump/iter_loc_edit_qwen/located/6_nli_located_38454.jsonl_filtered_0')

llm_edit = read_outputs('/data/hyeryung/mucoco/outputs/llmedit/results_saehee_2024/qwen/6_nli_edited_38454.jsonl_total_0')
llm_edit_wo = read_outputs('/data/hyeryung/mucoco/outputs/llmedit/results_saehee_2024/qwen/9_nli_loc_edit_38546.jsonl')

In [41]:
# constraint sat 파일 읽어오기
nli_result = read_metric_file('/data/hyeryung/mucoco/outputs/llmedit/results_2025/edited_nli/6_nli_edited_52941.jsonl_total_0-results.txt.nli', 'nli')
fluency_result = read_metric_file('/data/hyeryung/mucoco/outputs/llmedit/results_2025/edited_fluency/6_nli_edited_52941.jsonl_total_0-results.txt.fluency', 'fluency')
bertscore_result = read_metric_file('/data/hyeryung/mucoco/outputs/llmedit/results_2025/edited/6_nli_edited_52941.jsonl_total_0-results.txt.sbertscore', 'sbertscore')

llm_edit_sat = read_metric_file('/data/hyeryung/mucoco/outputs/llmedit/results_saehee_2024/qwen/nli_em_removed/6_nli_edited_38454.jsonl_total_0-results.txt.nli', 'nli')
llm_edit_wo_sat = read_metric_file('/data/hyeryung/mucoco/outputs/llmedit/results_saehee_2024/qwen/nli_em_removed/9_nli_loc_edit_38546.jsonl-results.txt.nli', 'nli')

In [42]:
# EDA 용 데이터프레임 생성
data_to_inspect = pd.DataFrame({'premise': original['prompt'],
              'original_h': original['text'],
              'edited_h': output['text'],
              'edited_h_for_nli': output_nli['text'],
              'edited_h_for_fluency': output_fluency['text'],
              'located': located['text'],
              'located_both': located_both['text'],
              'original_nli': original_sat['nli_class'],
              'edited_nli': nli_result['nli_class'],
              'edited_sbert': bertscore_result,
              'edited_fluency': fluency_result['fluency_proba'],
              'llm_edit_both': llm_edit['text'],
              'llm_edit_both_nli': llm_edit_sat['nli_class'],
              'llm_edit_wo': llm_edit_wo['text'],
              'llm_edit_wo_nli': llm_edit_wo_sat['nli_class']
              })


In [ ]:
# 보고 싶은 건은 고친 후에도 contradiction 인 건 과 그 중 원본 output 그대로 뱉은 건수와 output이 업데이트 된 건수가 얼마나 되는지 확인
# 확인 결과 : LLM Edit Masked가 contradiction이 높은건 원본을 그대로 뱉어서가 아니라 고쳤는데 잘못 고쳐서임
print('LLM Edit Masked')
print('# of contradictions: %d' % data_to_inspect.loc[data_to_inspect['edited_nli'] == 'contradiction'].shape[0])
print('# of contradictions that are output the same as original: %d' % data_to_inspect.loc[(data_to_inspect['edited_nli'] == 'contradiction') & 
                                        (data_to_inspect['original_h'] == data_to_inspect['edited_h'])].shape[0])
print('# of contradictions that are updated: %d' % data_to_inspect.loc[(data_to_inspect['edited_nli'] == 'contradiction') & 
                                        (data_to_inspect['original_h'] != data_to_inspect['edited_h'])].shape[0])

print('LLM Edit Both')
print('# of contradictions: %d' % data_to_inspect.loc[data_to_inspect['llm_edit_both_nli'] == 'contradiction'].shape[0])
print('# of contradictions that are output the same as original: %d' % data_to_inspect.loc[(data_to_inspect['llm_edit_both_nli'] == 'contradiction') & 
                                        (data_to_inspect['original_h'] == data_to_inspect['llm_edit_both'])].shape[0])
print('# of contradictions that are updated: %d' % data_to_inspect.loc[(data_to_inspect['llm_edit_both_nli'] == 'contradiction') & 
                                        (data_to_inspect['original_h'] != data_to_inspect['llm_edit_both'])].shape[0])

print('LLM Edit Without')
print('# of contradictions: %d' % data_to_inspect.loc[data_to_inspect['llm_edit_wo_nli'] == 'contradiction'].shape[0])
print('# of contradictions that are output the same as original: %d' % data_to_inspect.loc[(data_to_inspect['llm_edit_wo_nli'] == 'contradiction') & 
                                        (data_to_inspect['original_h'] == data_to_inspect['llm_edit_wo'])].shape[0])
print('# of contradictions that are updated: %d' % data_to_inspect.loc[(data_to_inspect['llm_edit_wo_nli'] == 'contradiction') & 
                                        (data_to_inspect['original_h'] != data_to_inspect['llm_edit_wo'])].shape[0])


'LLM Edit Masked'
'# of contradictions: 336'
'# of contradictions that are output the same as original: 130'
'# of contradictions that are updated: 206'
'LLM Edit Both'
'# of contradictions: 310'
'# of contradictions that are output the same as original: 143'
'# of contradictions that are updated: 167'
'LLM Edit Without'
'# of contradictions: 179'
'# of contradictions that are output the same as original: 17'
'# of contradictions that are updated: 162'


In [43]:
# 고친 output 들이 이상한지 확인
subset_to_inspect = data_to_inspect.loc[(data_to_inspect['edited_nli'] == 'contradiction') & 
                                        (data_to_inspect['original_h'] != data_to_inspect['edited_h']) & 
                                        (data_to_inspect['llm_edit_both_nli'] != 'contradiction') & 
                                        (data_to_inspect['llm_edit_wo_nli'] != 'contradiction')]

In [ ]:
# random sample 해서 보기  ## Locate 결과가 다름을 발견..
index = random.randint(0, subset_to_inspect.shape[0])
print(f'Premise: {subset_to_inspect.iloc[index]["premise"]}', width=130)
print(f'Original: {subset_to_inspect.iloc[index]["original_h"]}', width=130)
print(f'Located(Masked): {subset_to_inspect.iloc[index]["located"]}', width=130)
print(f'Edited(Masked): {subset_to_inspect.iloc[index]["edited_h"]}', width=130)
print(f'Located(Both): {subset_to_inspect.iloc[index]["located_both"]}', width=130)
print(f'Edited(Both): {subset_to_inspect.iloc[index]["llm_edit_both"]}', width=130)
print(f'Label(Both): {subset_to_inspect.iloc[index]["llm_edit_both_nli"]}', width=130)
print(f'Edited(Without): {subset_to_inspect.iloc[index]["llm_edit_wo"]}', width=130)
print(f'Label(Without): {subset_to_inspect.iloc[index]["llm_edit_wo_nli"]}', width=130)

('Premise: Big Sky is a census-designated place (CDP) in Gallatin and Madison counties in southwestern Montana. As of the 2010 '
 'census it had a population of 2,308. It is 45 mi southwest of Bozeman. This unincorporated community straddles the two '
 'counties, is not considered a town, and has no town government. The primary industry of the area is tourism.')
'Original: Big Sky is considered a town in southwestern Montana.'
'Located(Masked): Big Sky is considered a town in<mask> Montana.'
'Edited(Masked): Big Sky is considered a town in Gallatin and Madison counties.'
'Located(Both): Big Sky is<mask> a<mask> in<mask> Montana.'
'Edited(Both): Big Sky is not a town in Montana.'
'Label(Both): entail'
'Edited(Without): Big Sky is not considered a town in southwestern Montana.'
'Label(Without): entail'
